In [1]:
import sys
import sklearn
import os
import numpy as np

#Add the 'oracle' directory to the Python path
sys.path.append(os.path.join(os.getcwd(), 'oracle'))
import oracle
data = oracle.q2_train_test_emnist(23746,"EMNIST/emnist-balanced-train.csv","EMNIST/emnist-balanced-test.csv")
print(data[0].shape)
print(data[1].shape)
print(data[0])



/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


(4800, 785)
(800, 785)
[[17  0  0 ...  0  0  0]
 [17  0  0 ...  0  0  0]
 [17  0  0 ...  0  0  0]
 ...
 [30  0  0 ...  0  0  0]
 [30  0  0 ...  0  0  0]
 [17  0  0 ...  0  0  0]]


In [ ]:
def mean_variance(index,):
    data_17 = []
    data_30 = []
    for i in index :
        if data[0][i][0] == 17:
            data_17.append([np.longdouble(data[0][i][j]) for j in range(len(data[0][i]))])
        if data[0][i][0] == 30:
            data_30.append([np.longdouble(data[0][i][j]) for j in range(len(data[0][i]))])
    data_17 = np.array(data_17)
    data_30 = np.array(data_30)
    mean_list_17 = []
    std_list_17 = []
    mean_list_30 = []
    std_list_30 = []
    for i in range(1, 785):
        mean_list_17.append(np.mean(data_17[:, i]))
        std_list_17.append(np.std(data_17[:, i]))
        mean_list_30.append(np.mean(data_30[:, i]))
        std_list_30.append(np.std(data_30[:, i]))
    return mean_list_17,std_list_17,mean_list_30,std_list_30

def mean_cov():
    mean = np.mean(data[0], axis=0)
    cov = np.cov(data[0].T)
    return mean, cov

        
       
    


In [2]:
import math
import random

def transpose(matrix):
    """Returns the transpose of a matrix."""
    return list(map(list, zip(*matrix)))

def matmul(A, B):
    """Multiplies two matrices A and B."""
    return [[sum(A[i][k] * B[k][j] for k in range(len(B))) for j in range(len(B[0]))] for i in range(len(A))]

def identity(n):
    """Returns an identity matrix of size n x n."""
    return [[1 if i == j else 0 for j in range(n)] for i in range(n)]

def normalize(vec):
    """Normalizes a vector."""
    norm = math.sqrt(sum(x**2 for x in vec))
    return [x / norm for x in vec] if norm != 0 else vec

def power_iteration(A, num_iters=100, tol=1e-6):
    """Finds the largest eigenvalue and eigenvector of A using Power Iteration."""
    n = len(A)
    b_k = [random.random() for _ in range(n)]
    
    for _ in range(num_iters):
        b_k1 = matmul(A, [[x] for x in b_k])  # A * b_k
        b_k1 = [row[0] for row in b_k1]  # Convert from list of lists
        b_k1_norm = math.sqrt(sum(x**2 for x in b_k1))
        
        if b_k1_norm < tol:
            break

        b_k = [x / b_k1_norm for x in b_k1]  # Normalize

    # Rayleigh quotient (approximate eigenvalue)
    lambda_approx = sum(b_k[i] * matmul(A, [[b_k[i]] for i in range(n)])[i][0] for i in range(n))
    return lambda_approx, b_k

def eigen_decomposition(A, num_eigenvalues=None):
    """Computes eigenvalues and eigenvectors using Power Iteration + Deflation."""
    A_copy = [row[:] for row in A]  # Copy A to modify
    n = len(A)
    num_eigenvalues = num_eigenvalues or n
    
    eigenvalues = []
    eigenvectors = []

    for _ in range(num_eigenvalues):
        eigval, eigvec = power_iteration(A_copy)
        eigenvalues.append(eigval)
        eigenvectors.append(eigvec)

        # Deflate A to remove this eigenvector's contribution
        eigvec_outer = [[eigvec[i] * eigvec[j] for j in range(n)] for i in range(n)]
        A_copy = [[A_copy[i][j] - eigval * eigvec_outer[i][j] for j in range(n)] for i in range(n)]

    return eigenvalues, transpose(eigenvectors)

def svd_decomposition(A):
    """Computes the Singular Value Decomposition (SVD) of matrix A."""
    m, n = len(A), len(A[0])
    AT_A = matmul(transpose(A), A)  # A^T * A
    A_AT = matmul(A, transpose(A))  # A * A^T

    # Eigen decomposition of A^T * A
    eigvals_AT_A, U = eigen_decomposition(AT_A)
    
    # Sort eigenvalues and eigenvectors in descending order
    sorted_indices = sorted(range(len(eigvals_AT_A)), key=lambda k: eigvals_AT_A[k], reverse=True)
    U = [[U[row][i] for i in sorted_indices] for row in range(len(U))]
    eigvals_AT_A = [eigvals_AT_A[i] for i in sorted_indices]

    S = [[math.sqrt(val) if i == j else 0 for j in range(n)] for i, val in enumerate(eigvals_AT_A)]
    
    # Compute V from A * A^T eigen decomposition
    eigvals_A_AT, V = eigen_decomposition(A_AT)
    
    return U, S, V

def pseudo_inverse(A):
    """Computes the Moore-Penrose Pseudoinverse of a matrix using SVD."""
    U, S, V = svd_decomposition(A)
    
    # Compute S+ (pseudo-inverse of diagonal S)
    S_plus = [[1/S[i][i] if S[i][i] > 1e-10 else 0 for j in range(len(S))] for i in range(len(S))]
    
    # Compute A+ = V * S+ * U^T
    return matmul(matmul(V, S_plus), transpose(U))

# Example Usage:
A = [[2, 4], [1, 3], [0, 0]]
A_pinv = pseudo_inverse(A)

# Print result
print("Moore-Penrose Pseudoinverse of A:")
for row in A_pinv:
    print(row)


Moore-Penrose Pseudoinverse of A:
[-0.8789792276924245, 2.2735737130957268]
[1.0852856613885764, -2.807207352795605]
[0.0, 0.0]


In [ ]:
def Bayes(epsilon, x,mean_17,cov_17,mean_30,cov_30):
    p_17 = 1
    p_30 = 1
    cov_inverse_17 = pseudo_inverse(cov_17)
    cov_inverse_30 = pseudo_inverse(cov_30)
    p_17 = np.exp(-0.5 * np.dot(np.dot(np.transpose(x - mean_17), np.linalg.inv(cov_inverse_17)), x - mean_17))*(np.sqrt(np.linalg.det(cov_inverse_17)))
    p_30 = np.exp(-0.5 * np.dot(np.dot(np.transpose(x - mean_30), np.linalg.inv(cov_inverse_30)), x - mean_30))*(np.sqrt(np.linalg.det(cov_inverse_30))) 
    if p_17/(p_17+p_30) >= 0.5 + epsilon:
        return 17
    elif p_17/(p_17+p_30) <= 0.5 - epsilon:
        return 30
    return 0

def testing(index,values):
    TP = 0
    FP = 0
    TN = 0
    FN = 0
    rejection = 0
    for i in index:
        val = Bayes(0.25,data[0][i][1:],values[0],values[1],values[2],values[3])
        if val == 0:
            rejection += 1
            continue
        elif val == 17 and data[0][i][0] == 17:
            TP += 1
        elif val == 17 and data[0][i][0] == 30:
            FP += 1
        elif val == 30 and data[0][i][0] == 30:
            TN += 1
        elif val == 30 and data[0][i][0] == 17:
            FN += 1
    # print(misclassified)
    # print(rejection)
    return TP,FP,TN,FN,rejection


In [13]:
# Use KFold Bayes training
from sklearn.model_selection import KFold
average_miss = 0
kf = KFold(n_splits=5, shuffle=False)
miss = 0
best_values = []
for i, (train_index, test_index) in enumerate(kf.split(data[0])):
    print("Run", i)
    confusion_matrix = np.zeros((2,2))
    values = mean_variance(train_index)
    if i == 1 :
        best_values = values
    TP,FP,TN,FN,rejection = testing(test_index,values)
    confusion_matrix[0][0] = TP
    confusion_matrix[0][1] = FP
    confusion_matrix[1][0] = FN
    confusion_matrix[1][1] = TN
    print("Confusion:",confusion_matrix)
    recall = TP/(TP+FN)
    precision = TP/(TP+FP)
    f1 = 2*recall*precision/(recall+precision)
    accuracy = (TP+TN)/(TP+TN+FP+FN)
    print("Recall:", recall)
    print("Precision:", precision)
    print("F1 Score:", f1)
    print("Accuracy:", accuracy)
    print("Rejection Rate:", rejection)



Run 0
Confusion: [[413.  60.]
 [ 45. 442.]]
Recall: 0.9017467248908297
Precision: 0.8731501057082452
F1 Score: 0.8872180451127821
Accuracy: 0.890625
Rejection Rate: 0
Run 1


/tmp/ipykernel_131565/117129473.py:12: RuntimeWarning: invalid value encountered in scalar divide
  if p_17/(p_17+p_30) >= 0.5 + epsilon:
/tmp/ipykernel_131565/117129473.py:14: RuntimeWarning: invalid value encountered in scalar divide
  elif p_17/(p_17+p_30) <= 0.5 - epsilon:


Confusion: [[434.  41.]
 [ 43. 436.]]
Recall: 0.909853249475891
Precision: 0.9136842105263158
F1 Score: 0.911764705882353
Accuracy: 0.9119496855345912
Rejection Rate: 6
Run 2
Confusion: [[409.  42.]
 [ 56. 449.]]
Recall: 0.8795698924731182
Precision: 0.9068736141906873
F1 Score: 0.8930131004366813
Accuracy: 0.897489539748954
Rejection Rate: 4
Run 3
Confusion: [[448.  43.]
 [ 61. 406.]]
Recall: 0.8801571709233792
Precision: 0.9124236252545825
F1 Score: 0.896
Accuracy: 0.8914405010438413
Rejection Rate: 2
Run 4
Confusion: [[428.  53.]
 [ 51. 424.]]
Recall: 0.8935281837160751
Precision: 0.8898128898128899
F1 Score: 0.8916666666666667
Accuracy: 0.891213389121339
Rejection Rate: 4


In [16]:
def test_on_test_data(values):
    rejection = 0
    non_rejection = 0
    misclassified = 0
    for i in range(len(data[1])):
        val = Bayes(0.25,data[1][i][1:],values[0],values[1],values[2],values[3])
        if val == 0:
            rejection += 1
            continue
        else :
            non_rejection += 1
            if val != data[1][i][0]:
                misclassified += 1
    return misclassified, rejection, non_rejection

In [17]:
print("Testing on test data")
misclassified, rejection, non_rejection = test_on_test_data(best_values)
print("Misclassified:",misclassified)
print("Rejection:",rejection)
print("Non-Rejection:",non_rejection)
print("Misclassification Rate:",misclassified/non_rejection)

Testing on test data


/tmp/ipykernel_131565/117129473.py:12: RuntimeWarning: invalid value encountered in scalar divide
  if p_17/(p_17+p_30) >= 0.5 + epsilon:
/tmp/ipykernel_131565/117129473.py:14: RuntimeWarning: invalid value encountered in scalar divide
  elif p_17/(p_17+p_30) <= 0.5 - epsilon:


Misclassified: 89
Rejection: 5
Non-Rejection: 795
Misclassification Rate: 0.1119496855345912
